# corrector_03 - Transformer tagger + temperature scaling (role B)

Needs: `corrector_02_ngram.ipynb` already run (it saves `checkpoints/corrector/ngram.pkl.gz` and `viterbi.json`).

1. Run all unit tests (here PyTorch exists, so the transformer tests run too).
2. Build the tagger vocabulary from the n-gram model.
3. Train the tagger on noisy training sentences (new noise every epoch).
4. Temperature scaling on validation logits; compare NLL and ECE before and after.
5. Save `checkpoints/corrector/tagger.pt` and compare all methods on the same validation pairs.

Use a **GPU** runtime: Runtime > Change runtime type > T4 GPU. Needs the `GITHUB_TOKEN` secret.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GITHUB_TOKEN")
BRANCH = "b/corrector"  # change to "main" after the code is merged
!rm -rf /content/repo
!git clone -q -b {BRANCH} https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"

In [ ]:
!python -m unittest discover -s tests -t . -v 2>&1 | tail -8

In [ ]:
import itertools, json, os, time
import numpy as np
import torch

N_TRAIN = 600_000       # training sentences (new noise every epoch)
N_VAL = 5_000           # validation sentences for accuracy and temperature scaling
N_EVAL = 3_000          # validation sentences for the final comparison table
EPOCHS = 3
BATCH = 128
LR = 5e-4
CONFIG = {"d_model": 256, "n_heads": 4, "n_layers": 4, "d_ff": 1024, "max_len": 64, "dropout": 0.1}

CORPUS_DIR = f"{DATA_DIR}/corpus"
MODEL_DIR = f"{CKPT_DIR}/corrector"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def read_lines(path, n):
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in itertools.islice(f, n)]

train_sentences = read_lines(f"{CORPUS_DIR}/train.txt", N_TRAIN)
val_sentences = read_lines(f"{CORPUS_DIR}/val.txt", N_VAL)
print(len(train_sentences), "train,", len(val_sentences), "val sentences")

## Vocabulary

In [ ]:
from src.corrector.ngram import NgramModel
from src.corrector.tagger_data import TaggerVocab

ngram = NgramModel.load(f"{MODEL_DIR}/ngram.pkl.gz")
vocab = TaggerVocab.from_ngram(ngram, min_count=3, k=8)
print("forms:", len(vocab.forms), " keys:", len(vocab.keys))

## Train

In [ ]:
from torch.utils.data import DataLoader
from src.corrector.noise import NoiseConfig
from src.corrector.tagger import (DiacriticTagger, NoisySentences, collate, collect_logits,
                                  make_scheduler, save_checkpoint, train_epoch)

torch.manual_seed(0)
train_data = NoisySentences(train_sentences, vocab, NoiseConfig(), seed=1, max_len=CONFIG["max_len"])
val_data = NoisySentences(val_sentences, vocab, NoiseConfig(), seed=123, max_len=CONFIG["max_len"],
                          fixed_noise=True)
train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
val_loader = DataLoader(val_data, batch_size=256, collate_fn=collate, num_workers=2)

model = DiacriticTagger(len(vocab.forms), len(vocab.keys), **CONFIG).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = make_scheduler(optimizer, total_steps, warmup_steps=1000)

K_MAX = vocab.k + 1
os.makedirs(MODEL_DIR, exist_ok=True)
best_acc = 0.0
for epoch in range(EPOCHS):
    train_data.epoch = epoch  # new noise for this epoch
    start = time.time()
    loss = train_epoch(model, train_loader, optimizer, scheduler, device, log_every=1000)
    logits, targets = collect_logits(model, val_loader, device, K_MAX)
    acc = (logits.argmax(1) == targets).mean()
    print(f"epoch {epoch + 1}: train loss {loss:.4f}  val accuracy {acc:.4%}  ({time.time() - start:.0f} s)")
    if acc > best_acc:
        best_acc = acc
        save_checkpoint(f"{MODEL_DIR}/tagger.pt", model, vocab, CONFIG, temperature=1.0)
        print("  saved best model")

## Temperature scaling

Only positions with 2 or more candidates count: a syllable with one candidate is always 100 % sure.

In [ ]:
from src.corrector.calibration import calibration_report, fit_temperature
from src.corrector.tagger import load_checkpoint

best = load_checkpoint(f"{MODEL_DIR}/tagger.pt", device)
logits, targets = collect_logits(best.model, val_loader, device, K_MAX)
multi = np.isfinite(logits).sum(axis=1) >= 2
temperature = fit_temperature(logits[multi], targets[multi])
report = calibration_report(logits[multi], targets[multi], temperature)
print(json.dumps(report, indent=2))

save_checkpoint(f"{MODEL_DIR}/tagger.pt", best.model, vocab, CONFIG, temperature=temperature)
print("saved", f"{MODEL_DIR}/tagger.pt", "with T =", round(temperature, 3))

## Compare all methods on the same validation pairs

In [ ]:
from src.corrector import api
from src.corrector.evaluate import evaluate, print_table
from src.corrector.noise import make_pairs

# Older runs of corrector_02 did not save viterbi.json: make it here if it is missing.
if not os.path.exists(f"{MODEL_DIR}/viterbi.json"):
    from src.corrector.viterbi import estimate_p_same
    p_same = estimate_p_same(make_pairs(train_sentences[:20_000], seed=1))
    with open(f"{MODEL_DIR}/viterbi.json", "w", encoding="utf-8") as f:
        json.dump({"p_same": p_same, "k": 8}, f)

correctors ={name: api.load(MODEL_DIR, method=name, device=device)
              for name in ["identity", "frequency", "viterbi", "transformer"]}
settings = {"mixed noise": NoiseConfig(), "fully stripped": NoiseConfig(p_strip_sentence=1.0)}

eval_sentences = val_sentences[:N_EVAL]
results = {}
for setting, config in settings.items():
    pairs = list(make_pairs(eval_sentences, seed=123, config=config))  # same seed as corrector_02
    rows = {name: evaluate(c, pairs, ngram) for name, c in correctors.items()}
    print(f"\n=== {setting}  ({rows['identity']['n_syllables']} syllables) ===")
    print_table(rows)
    results[setting] = rows

os.makedirs(OUT_DIR, exist_ok=True)
with open(f"{OUT_DIR}/corrector_eval.json", "w", encoding="utf-8") as f:
    json.dump({"calibration": report, "config": CONFIG, "epochs": EPOCHS, "n_train": N_TRAIN,
               "results": results}, f, ensure_ascii=False, indent=2)
print("\nsaved", f"{OUT_DIR}/corrector_eval.json")

## The contract function, as the rest of the group will use it

In [ ]:
api.load(MODEL_DIR, method="transformer", device=device)
for text in ["nhieu ban doc da bay to", "Toi mua ban ghe moi", "cua hang ban hang rat dong",
             "Ngưới Việt Nam đã bày to sự quan tâm"]:
    out = api.correct(text)
    print(text, "->", out["text"])
    print("   conf:", [round(c, 2) for c in out["conf"]])